In [3]:
import pandas as pd

data_dir = "/Users/oliviakuang/Documents/GitHub/Cloudseeding2"

unmatched = pd.read_csv(
    f"{data_dir}/check/unmatched_operations.csv"
)

retained = pd.read_csv(
    f"{data_dir}/intermediate/operation_with_exact_times.csv"
)

# Correct the known swapped coordinates
swap = (
    unmatched["date"].eq("2022-10-27")
    & unmatched["lon"].eq(29.043)
    & unmatched["lat"].eq(115.560)
    & unmatched["city_o"].eq("九江市")
)

unmatched.loc[swap, ["lon", "lat"]] = [115.560, 29.043]

keys = ["date", "lon", "lat", "city_o", "county_o"]

review = unmatched.merge(
    retained[keys + ["cell_id"]].drop_duplicates(),
    on=keys,
    how="left",
)

review["retained_in_grid"] = review["cell_id"].notna()

print(
    review[
        keys + ["cell_id", "retained_in_grid"]
    ].to_string(index=False)
)

summary = (
    review.groupby(
        ["lon", "lat", "city_o", "county_o", "cell_id", "retained_in_grid"],
        dropna=False,
    )
    .size()
    .reset_index(name="number_of_records")
)

print(summary.to_string(index=False))

      date       lon      lat city_o county_o cell_id  retained_in_grid
2021-09-06 115.38100 24.56500    赣州市      寻乌县     NaN             False
2022-10-27 115.56000 29.04300    九江市      永修县  100_39              True
2023-07-31 115.76900 24.81400    赣州市      寻乌县    6_42              True
2023-07-30 115.76900 24.81400    赣州市      寻乌县    6_42              True
2023-07-18 115.76900 24.81400    赣州市      寻乌县    6_42              True
2023-07-18 115.76900 24.81400    赣州市      寻乌县    6_42              True
2023-07-17 115.76900 24.81400    赣州市      寻乌县    6_42              True
2023-07-16 115.76900 24.81400    赣州市      寻乌县    6_42              True
2023-03-11 118.64000 28.49000    上饶市      广丰区     NaN             False
2023-03-11 118.64000 28.49000    上饶市      广丰区     NaN             False
2024-12-07 115.76900 24.81400    赣州市      寻乌县    6_42              True
2024-11-14 115.76900 24.81400    赣州市      寻乌县    6_42              True
2024-11-13 115.76900 24.81400    赣州市      寻乌县    6_42           

Among the 31 records that did not fall strictly within a township polygon(potentially due to rounding errors, falling exactly on the township boundaries, or coordinate errors), 26 are retained because their calculated 5 km grid cells overlap Jiangxi. The corrected longitude–latitude swap is assigned to valid cell 100_39; the 24 repeated 寻乌 records are assigned to valid cell 6_42; and the 浮梁 record is assigned to valid cell 113_80. Note that their retention does not mean that the exact points lie within Jiangxi, because a boundary grid cell may extend beyond the provincial or township boundaries.

The other five unmatched records are excluded because their grid cells do not overlap Jiangxi: one 寻乌 record at (115.381, 24.565), two 广丰 records at (118.640, 28.490), and two 广昌 records at (116.34833, 25.84472) and (116.34833, 25.66111).

This review suggests that the only observation with a clearly identifiable data error is the known longitude–latitude swap, which can be corrected with confidence. The remaining unmatched records do not provide sufficient evidence for manual correction. Although the 5 excluded records can be further reviewed using external/additional sources, no defensible corrections can be made at the current stage. Therefore, they are retained or excluded according to the existing grid-cell filtering rule, and their unmatched status is documented for transparency.